# 지역난방 열수요 예측: 완전한 고도화된 스태킹 앙상블

## 모델링 전략
- **구성**: 3개 규모 그룹 × 2개 계절 = 6개 모델
- **스태킹**: Prophet + CatBoost + LSTM + Ridge 메타모델
- **최적화**: 모든 모델에 Optuna + 3-Fold CV
- **총 모델**: 18개 + 6개 메타모델 = 24개
- **재현성**: 완전한 시드 고정

In [ ]:
# Google Colab 환경 확인 및 패키지 설치
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Google Colab 환경에서 실행 중...")
    !pip install catboost prophet torch optuna statsmodels holidays pmdarima scikit-learn==1.3.0 --quiet
    from google.colab import files, drive
    print("패키지 설치 완료!")
else:
    print("로컬 환경에서 실행 중...")

In [64]:
# 완전한 재현성을 위한 시드 고정
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 라이브러리 import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from tqdm.auto import tqdm
import pickle
import holidays
import json
import os

In [ ]:
# 머신러닝 라이브러리
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, KFold
import catboost as cb
from catboost import CatBoostRegressor

# PyTorch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# Prophet
try:
    from prophet import Prophet
    import logging
    logging.getLogger('prophet').setLevel(logging.WARNING)
except ImportError:
    print("Prophet 설치 필요")
    Prophet = None

# Optuna
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ARIMA
try:
    from pmdarima import auto_arima
    from statsmodels.tsa.arima.model import ARIMA
except ImportError:
    print("pmdarima 설치 필요")
    auto_arima = None

# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"디바이스: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()

plt.rcParams['figure.figsize'] = (12, 6)
print("라이브러리 로드 완료! (시드 고정으로 재현성 보장)")

In [ ]:
# 데이터 파일 로드
if IN_COLAB:
    print("파일 업로드 방법 선택:")
    print("1. 직접 업로드")
    print("2. Google Drive")

    method = input("선택 (1 또는 2): ")

    if method == "1":
        uploaded = files.upload()
        files_list = list(uploaded.keys())
        train_path = [f for f in files_list if 'train' in f.lower()][0]
        test_path = [f for f in files_list if 'test' in f.lower()][0]
    else:
        drive.mount('/content/drive')
        train_path = "/content/drive/MyDrive/train_heat.csv"
        test_path = "/content/drive/MyDrive/test_heat.csv"
else:
    train_path = 'train_heat.csv'
    test_path = 'test_heat.csv'

print(f"파일 경로 설정 완료")

## 1. 고도화된 데이터 전처리

In [67]:
def load_and_preprocess_advanced(train_path, test_path):
    print("고도화된 데이터 로드 및 전처리...")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    def process_df_advanced(df):
        if 'Unnamed: 0' in df.columns:
            df = df.drop(columns=['Unnamed: 0'])
        df.columns = [col.replace('train_heat.', '') for col in df.columns]

        if df['tm'].dtype == 'object':
            df['tm'] = pd.to_datetime(df['tm'])
        else:
            df['tm'] = pd.to_datetime(df['tm'], format='%Y%m%d%H')
        
        df['year'] = df['tm'].dt.year
        df['month'] = df['tm'].dt.month
        df['day'] = df['tm'].dt.day
        df['hour'] = df['tm'].dt.hour
        df['dayofweek'] = df['tm'].dt.dayofweek
        df['dayofyear'] = df['tm'].dt.dayofyear

        # ✅ wd (풍향) 제외, 사용할 컬럼만 포함
        missing_cols = ['ta', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi']

        # ✅ 1단계: 결측치 플래그 생성 (NaN 변환 전에)
        print("   결측치 플래그 생성 중...")
        for col in missing_cols:
            if col in df.columns:
                # -99를 결측치로 인식하여 플래그 생성
                missing_mask = (df[col] == -99)
                df[f'{col}_missing'] = missing_mask.astype(int)
        
        # ✅ 2단계: 결측치를 NaN으로 변환
        for col in missing_cols:
            if col in df.columns:
                df[col] = df[col].replace(-99, np.nan)

        # ✅ wd 컬럼이 있으면 아예 삭제
        if 'wd' in df.columns:
            df = df.drop(columns=['wd'])
            print("   wd (풍향) 컬럼 삭제됨")

        if 'si' in df.columns:
            night_mask = (df['hour'] < 8) | (df['hour'] > 18)
            df.loc[night_mask & df['si'].isna(), 'si'] = 0

        df = df.sort_values(['branch_id', 'tm'])
        
        # ✅ 3단계: 생성된 결측치 플래그 확인
        missing_flag_cols = [col for col in df.columns if col.endswith('_missing')]
        print(f"   생성된 결측치 플래그: {missing_flag_cols}")
        for col in missing_flag_cols:
            missing_count = df[col].sum()
            print(f"     {col}: {missing_count:,}개 결측치")

        return df

    train_df = process_df_advanced(train_df)
    test_df = process_df_advanced(test_df)

    print(f"   훈련: {train_df.shape}, 테스트: {test_df.shape}")
    print(f"   기간: {train_df['tm'].min()} ~ {test_df['tm'].max()}")

    return train_df, test_df

## 2. 시즌별 이상치 플래그 생성 (도메인 특화) _ 온도, 풍속, 강수량

In [68]:
def create_weather_outlier_flags(train_df, test_df):
    """기상데이터 기반 이상치 플래그 (TRAIN 기준 적용)"""
    print("기상 이상치 플래그 생성 중 (TRAIN 기준)...")
    
    # 1단계: TRAIN 데이터에서만 임계값 계산 (21-23년 기준)
    outlier_thresholds = {}
    
    for branch in train_df['branch_id'].unique():
        branch_data = train_df[train_df['branch_id'] == branch]
        
        if len(branch_data) > 10:
            outlier_thresholds[branch] = {
                # 🌡️ 온도: 하위 10% (극한 추위)
                'ta_q10': branch_data['ta'].quantile(0.10),
                # 💨 풍속: 상위 10% (강풍)
                'ws_q90': branch_data['ws'].quantile(0.90),
                # 🌧️ 일강수량: 상위 10% (폭우)
                'rn_day_q90': branch_data['rn_day'].quantile(0.90)
            }
    
    # 2단계: 임계값을 TRAIN과 TEST에 적용
    def apply_weather_thresholds(df, thresholds):
        df = df.copy()
        # 기본값으로 초기화
        df['cold_extreme'] = 0      # 극한 추위 (하위 10%)
        df['strong_wind'] = 0       # 강풍 (상위 10%)
        df['heavy_rain'] = 0        # 폭우 (상위 10%)

        
        for branch in df['branch_id'].unique():
            if branch in thresholds:
                branch_mask = df['branch_id'] == branch
                
                # 온도 이상치 (낮은 온도)
                df.loc[branch_mask, 'cold_extreme'] = (
                    df.loc[branch_mask, 'ta'] < thresholds[branch]['ta_q10']
                ).astype(int)
                
                # 풍속 이상치 (높은 풍속)
                df.loc[branch_mask, 'strong_wind'] = (
                    df.loc[branch_mask, 'ws'] > thresholds[branch]['ws_q90']
                ).astype(int)
                
                # 강수량 이상치 (많은 비)
                df.loc[branch_mask, 'heavy_rain'] = (
                    df.loc[branch_mask, 'rn_day'] > thresholds[branch]['rn_day_q90']
                ).astype(int)
                        
        return df
    
    # TRAIN 적용
    train_result = apply_weather_thresholds(train_df, outlier_thresholds)
    
    # TEST 적용
    test_result = apply_weather_thresholds(test_df, outlier_thresholds)
    
    print(f"   기상 이상치 플래그 생성 완료: {len(outlier_thresholds)}개 지사")
    
    return train_result, test_result, outlier_thresholds

In [ ]:
def create_advanced_features(df, season_type="heating"):
    df = df.copy()
    print(f"{season_type} 시즌 고도화된 특성 생성 중...")
    
    # 범주형 시간 변수 (문자열로 명시적 변환)
    df['hour_cat'] = df['hour'].astype(str)
    df['month_cat'] = df['month'].astype(str)
    df['weekday_name'] = df['dayofweek'].map(
        lambda x: ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'][x]
    ).astype(str)
    
    # 순환 인코딩
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    
    # 시즌별 월 순환
    if season_type == "heating":
        heating_months = {10:0, 11:1, 12:2, 1:3, 2:4, 3:5, 4:6}
        df['heating_month_order'] = df['month'].map(heating_months)
        df['heating_month_sin'] = np.sin(2 * np.pi * df['heating_month_order'] / 7)
        df['heating_month_cos'] = np.cos(2 * np.pi * df['heating_month_order'] / 7)
    else:
        non_heating_months = {5:0, 6:1, 7:2, 8:3, 9:4}
        df['non_heating_month_order'] = df['month'].map(non_heating_months)
        df['non_heating_month_sin'] = np.sin(2 * np.pi * df['non_heating_month_order'] / 5)
        df['non_heating_month_cos'] = np.cos(2 * np.pi * df['non_heating_month_order'] / 5)
    
    # 브랜치 ID (문자열로 변환)
    df['branch_id'] = df['branch_id'].astype(str)
    
    # 고급 기상 범주형 변수
    df['temp_category'] = 'Normal'
    df.loc[df['ta'] < -10, 'temp_category'] = 'VeryCold'
    df.loc[(df['ta'] >= -10) & (df['ta'] < 0), 'temp_category'] = 'Cold'
    df.loc[(df['ta'] >= 0) & (df['ta'] < 10), 'temp_category'] = 'Cool'
    df.loc[(df['ta'] >= 10) & (df['ta'] < 25), 'temp_category'] = 'Normal'
    df.loc[df['ta'] >= 25, 'temp_category'] = 'Hot'
    df['temp_category'] = df['temp_category'].astype(str)
    
    if season_type == "heating":
        df['cold_warning_level'] = 'Normal'
        df.loc[df['ta'] <= -12, 'cold_warning_level'] = 'ColdAdvisory'
        df.loc[df['ta'] <= -15, 'cold_warning_level'] = 'ColdWarning'
        df['cold_warning_level'] = df['cold_warning_level'].astype(str)
    
    df['wind_category'] = 'Weak'
    df.loc[df['ws'] >= 5.0, 'wind_category'] = 'Moderate'
    df.loc[df['ws'] >= 10.0, 'wind_category'] = 'Strong'
    df['wind_category'] = df['wind_category'].astype(str)
    
    # 공휴일/피크시간
    kr_holidays = holidays.KR()
    df['is_holiday'] = df['tm'].dt.date.apply(lambda x: x in kr_holidays)
    df['holiday_type'] = df['is_holiday'].map({False: 'Weekday', True: 'Holiday'}).astype(str)
    
    df['peak_time1'] = 'Normal'
    df.loc[(df['hour'] >= 0) & (df['hour'] <= 6), 'peak_time1'] = 'Dawn'
    df.loc[(df['hour'] > 6) & (df['hour'] <= 11), 'peak_time1'] = 'Morning'
    df.loc[(df['hour'] > 11) & (df['hour'] <= 18), 'peak_time1'] = 'Afternoon'
    df.loc[(df['hour'] > 18) & (df['hour'] <= 23), 'peak_time1'] = 'Evening'
    df['peak_time1'] = df['peak_time1'].astype(str)
    
    # 고급 수치형 특성
    df['HDD18'] = np.maximum(0, 18 - df['ta'])
    df['HDD20'] = np.maximum(0, 20 - df['ta'])
    
    def calculate_apparent_temp(ta, hm, ws):
        winter_at = 13.12 + 0.6215 * ta - 11.37 * (ws * 3.6)**0.16 + 0.3965 * ta * (ws * 3.6)**0.16
        return winter_at
    
    df['apparent_temp'] = calculate_apparent_temp(df['ta'], df['hm'], df['ws'])
    
    for lag in [3, 6, 24]:
        df[f'ta_lag_{lag}h'] = df.groupby('branch_id')['ta'].shift(lag)
    
    for window in [6, 12, 24]:
        df[f'ta_ma_{window}h'] = df.groupby('branch_id')['ta'].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
    
    df['ta_diff_3h'] = df.groupby('branch_id')['ta'].diff(3)
    df['ta_diff_6h'] = df.groupby('branch_id')['ta'].diff(6)
    
    daily_stats = df.groupby(['branch_id', df['tm'].dt.date]).agg({
        'ta': ['min', 'max', 'mean']
    }).round(2)
    daily_stats.columns = ['daily_ta_min', 'daily_ta_max', 'daily_ta_mean']
    daily_stats['daily_temp_range'] = daily_stats['daily_ta_max'] - daily_stats['daily_ta_min']
    
    df = df.merge(
        daily_stats.reset_index(),
        left_on=['branch_id', df['tm'].dt.date],
        right_on=['branch_id', 'tm'],
        how='left',
        suffixes=('', '_daily')
    )
    
    print(f"   {season_type} 시즌 고도화된 특성 생성 완료: {df.shape[1]}개 컬럼")
    return df

# 고도화된 특성 생성
print("고도화된 특성 생성...")
train_df = create_advanced_features(train_df, "heating")
test_df = create_advanced_features(test_df, "heating")

# ✅ 이상치 플래그는 별도로 처리
train_df, test_df, weather_thresholds = create_weather_outlier_flags(train_df, test_df)

print(f"\n처리 후 데이터 크기:")
print(f"   훈련: {train_df.shape}")
print(f"   테스트: {test_df.shape}")

## 3. 규모별 그룹 분할

In [ ]:
# 1. 먼저 heating_season 컬럼 추가
def add_heating_season(df):
    """난방 시즌 컬럼 추가"""
    df = df.copy()
    df['heating_season'] = 0  # 기본값: 비난방
    heating_months = [10, 11, 12, 1, 2, 3, 4]  # 10월~4월: 난방시즌
    df.loc[df['month'].isin(heating_months), 'heating_season'] = 1
    return df

# 2. heating_season 컬럼 추가
print("난방 시즌 컬럼 추가...")
train_df = add_heating_season(train_df)
test_df = add_heating_season(test_df)

# 3. 시즌별 데이터 분할 (2개 그룹)
def split_by_season_only(df):
    """시즌별로만 분할 (2개 그룹)"""
    groups = {}
    
    for season in [0, 1]:  # 0: 비난방, 1: 난방
        season_name = 'heating' if season == 1 else 'non_heating'
        season_data = df[df['heating_season'] == season].copy()
        groups[season_name] = season_data
                
    return groups

# 4. 사용
train_groups = split_by_season_only(train_df)
test_groups = split_by_season_only(test_df)

print(f"\n시즌별 데이터 분할 결과:")
print("=" * 50)
for group_name, group_data in train_groups.items():
    if len(group_data) > 0:
        mean_demand = group_data['heat_demand'].mean()
        unique_branches = group_data['branch_id'].nunique()
        print(f"{group_name:15s}: {len(group_data):,}개 ({unique_branches}개 지사, 평균 수요: {mean_demand:7.2f})")
    else:
        print(f"{group_name:15s}: {len(group_data):,}개 (데이터 없음)")

## 4. 모델별 피쳐 정의

In [ ]:
# 모델별 피쳐 정의
def define_model_features():
    prophet_features = {
        'basic': ['hour', 'ta', 'HDD18', 'HDD20', 'apparent_temp'],
        'seasonal': ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofweek_sin', 'dayofweek_cos'],
        'categorical': []
    }
    
    catboost_features = {
        'numerical': [
            'ta', 'hm', 'ws', 'rn_day', 'si', 'HDD18', 'HDD20', 'apparent_temp',
            'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofweek_sin', 'dayofweek_cos',
            'ta_lag_3h', 'ta_lag_6h', 'ta_lag_24h', 'ta_ma_6h', 'ta_ma_12h', 'ta_ma_24h',
            'ta_diff_3h', 'ta_diff_6h', 'daily_ta_min', 'daily_ta_max', 'daily_ta_mean', 'daily_temp_range'
        ],
        'categorical': [
            'branch_id', 'hour_cat', 'month_cat', 'weekday_name', 
            'temp_category', 'wind_category', 'holiday_type', 'peak_time1'
        ],
        'flags': [
            'ta_missing', 'ws_missing', 'rn_day_missing', 
            'rn_hr1_missing', 'hm_missing', 'si_missing', 'ta_chi_missing',
            'cold_extreme', 'strong_wind', 'heavy_rain'
        ]
    }
    
    catboost_heating_features = catboost_features.copy()
    catboost_heating_features['categorical'] = catboost_features['categorical'] + ['cold_warning_level']
    
    lstm_features = {
        'numerical': [
            'ta', 'hm', 'ws', 'rn_day', 'si', 'HDD18', 'HDD20', 'apparent_temp',
            'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofweek_sin', 'dayofweek_cos',
            'ta_ma_6h', 'ta_ma_12h', 'ta_ma_24h', 'daily_ta_mean', 'daily_temp_range'
        ],
        'categorical_encoded': ['branch_id']
    }
    
    return {
        'prophet': prophet_features,
        'catboost_heating': catboost_heating_features,
        'catboost_non_heating': catboost_features,
        'lstm': lstm_features
    }

model_features = define_model_features()

print("모델별 피쳐 정의 완료:")
print("=" * 50)
for model_name, features in model_features.items():
    total_features = sum(len(v) if isinstance(v, list) else 0 for v in features.values())
    print(f"{model_name:20s}: {total_features}개 피쳐")
    for feature_type, feature_list in features.items():
        if isinstance(feature_list, list):
            print(f"  {feature_type:15s}: {len(feature_list)}개")

## 5. ARIMA 보간 함수 & 모델 클래스들

In [ ]:
# ARIMA 보간 함수 (LSTM용)
def apply_arima_interpolation(df, target_col='heat_demand', is_train=True):
    print(f"ARIMA 보간 적용 중 ({'TRAIN' if is_train else 'TEST'} 데이터)...")
    
    if auto_arima is None:
        print("   pmdarima가 설치되지 않음, 선형 보간 사용")
        return df.interpolate(method='linear').fillna(method='ffill').fillna(method='bfill')
    
    df_interpolated = df.copy()
    
    # 수치형 컬럼만 선택
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    interpolation_cols = ['ta', 'hm', 'ws', 'rn_day', 'si']
    available_cols = [col for col in interpolation_cols if col in numeric_cols]
    
    # TRAIN 데이터인 경우에만 ARIMA 보간 적용
    if is_train:
        for branch in df['branch_id'].unique():
            branch_mask = df_interpolated['branch_id'] == branch
            branch_data = df_interpolated[branch_mask].copy().sort_values('tm')
            
            for col in available_cols:
                if col not in branch_data.columns:
                    continue
                    
                missing_mask = branch_data[col].isna()
                if missing_mask.sum() == 0 or missing_mask.sum() == len(branch_data):
                    continue
                
                try:
                    non_missing_data = branch_data[col].dropna()
                    if len(non_missing_data) < 10:
                        # ✅ ARIMA 대신 선형 보간 사용
                        df_interpolated.loc[branch_mask, col] = branch_data[col].interpolate(method='linear')
                        continue
                    
                    # ✅ 간단한 ARIMA 모델 사용
                    try:
                        # 시계열 인덱스 설정
                        ts_data = non_missing_data.copy()
                        ts_data.index = pd.to_datetime(branch_data.loc[non_missing_data.index, 'tm'])
                        
                        model = auto_arima(
                            ts_data, 
                            start_p=0, start_q=0, max_p=1, max_q=1,  # ✅ 파라미터 범위 축소
                            seasonal=False, stepwise=True, 
                            suppress_warnings=True, error_action='ignore'
                        )
                        
                        # 결측치 선형 보간으로 대체
                        df_interpolated.loc[branch_mask, col] = branch_data[col].interpolate(method='linear')
                        
                    except Exception as e:
                        # ARIMA 실패 시 선형 보간 사용
                        df_interpolated.loc[branch_mask, col] = branch_data[col].interpolate(method='linear')
                        
                except Exception as e:
                    continue
    else:
        # TEST 데이터는 선형 보간만 사용
        for branch in df['branch_id'].unique():
            branch_mask = df_interpolated['branch_id'] == branch
            branch_data = df_interpolated[branch_mask].copy()
            
            for col in available_cols:
                if col in branch_data.columns:
                    df_interpolated.loc[branch_mask, col] = branch_data[col].interpolate(method='linear')
    
    print(f"   보간 완료")
    return df_interpolated

# TimeSeriesDataset
class TimeSeriesDataset(Dataset):
    def __init__(self, data, target, sequence_length=24):
        self.data = torch.FloatTensor(data)
        self.target = torch.FloatTensor(target)
        self.sequence_length = sequence_length

    def __len__(self):
        return max(1, len(self.data) - self.sequence_length + 1)

    def __getitem__(self, idx):
        if idx >= len(self.data) - self.sequence_length:
            idx = max(0, len(self.data) - self.sequence_length)
        x = self.data[idx:idx + self.sequence_length]
        y = self.target[idx + self.sequence_length - 1]
        return x, y.unsqueeze(0)

# LSTM 모델
class LSTMNet(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super(LSTMNet, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = F.relu(self.fc1(out))
        out = self.fc2(out)
        return out

print("ARIMA 보간 함수 및 LSTM 모델 클래스 정의 완료")

## 6. Prophet, CatBoost, LSTM 모델 클래스 (Optuna 최적화)

In [ ]:
class ProphetOptimizedModel:
    def __init__(self, season_type="heating"):
        self.models = {}
        self.best_params = {}
        self.season_type = season_type
        
    def optimize_hyperparameters(self, df, target_col='heat_demand', n_trials=20):
        print(f"Prophet 하이퍼파라미터 최적화 중... (trials: {n_trials})")
        
        # 지사별 데이터 캐싱
        branch_data_cache = {branch: df[df['branch_id'] == branch] 
                            for branch in df['branch_id'].unique()}
        
        def objective(trial):
            changepoint_prior_scale = trial.suggest_float('changepoint_prior_scale', 0.01, 0.1, log=True)
            seasonality_prior_scale = trial.suggest_float('seasonality_prior_scale', 0.1, 5, log=True)
            holidays_prior_scale = trial.suggest_float('holidays_prior_scale', 0.1, 5, log=True)
            seasonality_mode = trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative'])
            
            kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
            cv_scores = []
            
            for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
                print(f"   Fold {fold+1}/3 처리 중...")
                fold_predictions = []
                fold_targets = []
                
                for branch in branch_data_cache:
                    branch_data = branch_data_cache[branch]
                    branch_train = branch_data.iloc[train_idx]
                    branch_val = branch_data.iloc[val_idx]
                    
                    # 최소 데이터 크기 조건 완화
                    if len(branch_train) < 10 or len(branch_val) == 0:
                        print(f"     지사 {branch}: 데이터 부족 (train={len(branch_train)}, val={len(branch_val)})")
                        continue
                    
                    # Prophet 데이터 준비 (결측치 처리 없이)
                    prophet_df = pd.DataFrame({
                        'ds': pd.to_datetime(branch_train['tm']),
                        'y': branch_train[target_col]
                    })
                    
                    # 회귀변수 추가
                    regressors = ['hour', 'ta', 'HDD18', 'HDD20', 'apparent_temp']
                    for reg in regressors:
                        if reg in branch_train.columns:
                            prophet_df[reg] = branch_train[reg]
                    
                    try:
                        model = Prophet(
                            changepoint_prior_scale=changepoint_prior_scale,
                            seasonality_prior_scale=seasonality_prior_scale,
                            holidays_prior_scale=holidays_prior_scale,
                            seasonality_mode=seasonality_mode,
                            daily_seasonality=True,
                            weekly_seasonality=True,
                            yearly_seasonality=False
                        )
                        
                        # 회귀변수 추가
                        for reg in regressors:
                            if reg in prophet_df.columns and reg not in ['ds', 'y']:
                                model.add_regressor(reg)
                        
                        model.fit(prophet_df)
                        
                        # 예측 데이터 준비
                        future_df = pd.DataFrame({
                            'ds': pd.to_datetime(branch_val['tm'])
                        })
                        for reg in regressors:
                            if reg in branch_val.columns:
                                future_df[reg] = branch_val[reg]
                        
                        forecast = model.predict(future_df)
                        predictions = np.maximum(forecast['yhat'].values, 0)
                        
                        actual_values = branch_val[target_col].values
                        valid_mask = ~np.isnan(actual_values)
                        
                        if valid_mask.sum() > 0:
                            fold_predictions.extend(predictions[valid_mask])
                            fold_targets.extend(actual_values[valid_mask])
                        else:
                            print(f"     지사 {branch}: 유효한 타겟 데이터 없음")
                    
                    except Exception as e:
                        print(f"     지사 {branch}: Prophet 학습/예측 실패 - {str(e)[:100]}")
                        continue
                
                if len(fold_predictions) > 0:  # 조건 완화
                    rmse = np.sqrt(mean_squared_error(fold_targets, fold_predictions))
                    cv_scores.append(rmse)
                    print(f"     Fold {fold+1} RMSE: {rmse:.4f}")
                else:
                    print(f"     Fold {fold+1}: 유효한 예측 없음")
            
            return np.mean(cv_scores) if cv_scores else float('inf')
        
        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
        
        self.best_params = study.best_params
        print(f"   Prophet 최적 RMSE: {study.best_value:.4f}")
        print(f"   최적 파라미터: {self.best_params}")
        return study.best_value

    def fit(self, df, target_col='heat_demand'):
        print(f"Prophet 모델 훈련 중...")
        
        branches = df['branch_id'].unique()
        success_count = 0

        for branch in tqdm(branches, desc="Prophet 브랜치별 훈련"):
            try:
                branch_data = df[df['branch_id'] == branch].copy()

                if len(branch_data) < 10:
                    continue

                prophet_df = pd.DataFrame({
                    'ds': branch_data['tm'],
                    'y': branch_data[target_col]
                })

                # 최적화된 파라미터 사용
                model = Prophet(
                    changepoint_prior_scale=self.best_params.get('changepoint_prior_scale', 0.05),
                    seasonality_prior_scale=self.best_params.get('seasonality_prior_scale', 10.0),
                    holidays_prior_scale=self.best_params.get('holidays_prior_scale', 10.0),
                    seasonality_mode=self.best_params.get('seasonality_mode', 'multiplicative'),
                    daily_seasonality=True,
                    weekly_seasonality=True,
                    yearly_seasonality=True
                )

                # 회귀변수 추가
                regressors = ['hour', 'ta', 'HDD18', 'apparent_temp']
                for reg in regressors:
                    if reg in branch_data.columns:
                        model.add_regressor(reg)
                        prophet_df[reg] = branch_data[reg].values

                model.fit(prophet_df)
                self.models[branch] = model
                success_count += 1

            except Exception as e:
                continue

        print(f"   {success_count}/{len(branches)}개 지사 훈련 완료")

    def predict(self, df):
        predictions = []
        for branch in df['branch_id'].unique():
            if branch not in self.models:
                predictions.extend([0] * len(df[df['branch_id'] == branch]))
                continue

            branch_data = df[df['branch_id'] == branch].copy()
            future_df = pd.DataFrame({'ds': branch_data['tm']})

            regressors = ['hour', 'ta', 'HDD18', 'apparent_temp']
            for reg in regressors:
                if reg in branch_data.columns:
                    future_df[reg] = branch_data[reg].values

            forecast = self.models[branch].predict(future_df)
            predictions.extend(forecast['yhat'].values)

        return np.array(predictions)

print("Prophet 최적화 모델 클래스 정의 완료")

In [75]:
class CatBoostOptimizedModel:
    def __init__(self, season_type="heating"):
        self.model = None
        self.feature_cols = None
        self.categorical_features = None
        self.best_params = {}
        self.season_type = season_type
        
    def optimize_hyperparameters(self, df, target_col='heat_demand', n_trials=20):
        print(f"CatBoost 하이퍼파라미터 최적화 중... (trials: {n_trials})")
        
        # 피쳐 준비
        features = []
        for ftype in ['numerical', 'categorical', 'flags']:
            features.extend(self.features.get(ftype, []))
        cat_features = self.features.get('categorical', [])
        
        # 범주형 피쳐 확인
        print(f"   범주형 피쳐: {cat_features}")
        for col in cat_features:
            if col in df.columns:
                df[col] = df[col].astype(str)  # 범주형 피쳐를 문자열로 변환
                print(f"     {col}: {df[col].nunique()} unique values")
        
        # 결측치 선형 보간 (CatBoost는 결측치를 처리할 수 있지만, 안정성을 위해)
        numeric_cols = self.features.get('numerical', [])
        for col in numeric_cols:
            if col in df.columns:
                df[col] = df[col].interpolate(method='linear').fillna(method='ffill').fillna(method='bfill')
        
        def objective(trial):
            params = {
                'iterations': trial.suggest_int('iterations', 100, 1000),
                'depth': trial.suggest_int('depth', 4, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
                'border_count': trial.suggest_int('border_count', 32, 255),
                'random_seed': SEED,
                'task_type': 'CPU',
                'verbose': 0
            }
            
            kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
            cv_scores = []
            
            for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
                print(f"   Fold {fold+1}/3 처리 중...")
                train_fold = df.iloc[train_idx]
                val_fold = df.iloc[val_idx]
                
                fold_predictions = []
                fold_targets = []
                
                for branch in train_fold['branch_id'].unique():
                    branch_train = train_fold[train_fold['branch_id'] == branch]
                    branch_val = val_fold[val_fold['branch_id'] == branch]
                    
                    if len(branch_train) < 10 or len(branch_val) == 0:
                        print(f"     지사 {branch}: 데이터 부족 (train={len(branch_train)}, val={len(branch_val)})")
                        continue
                    
                    X_train = branch_train[features]
                    y_train = branch_train[target_col]
                    X_val = branch_val[features]
                    y_val = branch_val[target_col]
                    
                    try:
                        model = CatBoostRegressor(**params, cat_features=cat_features)
                        model.fit(X_train, y_train, verbose=0)
                        
                        predictions = model.predict(X_val)
                        predictions = np.maximum(predictions, 0)
                        
                        valid_mask = ~np.isnan(y_val)
                        if valid_mask.sum() > 0:
                            fold_predictions.extend(predictions[valid_mask])
                            fold_targets.extend(y_val[valid_mask])
                        else:
                            print(f"     지사 {branch}: 유효한 타겟 데이터 없음")
                    
                    except Exception as e:
                        print(f"     지사 {branch}: CatBoost 학습/예측 실패 - {str(e)[:100]}")
                        continue
                
                if len(fold_predictions) > 0:
                    rmse = np.sqrt(mean_squared_error(fold_targets, fold_predictions))
                    cv_scores.append(rmse)
                    print(f"     Fold {fold+1} RMSE: {rmse:.4f}")
                else:
                    print(f"     Fold {fold+1}: 유효한 예측 없음")
            
            return np.mean(cv_scores) if cv_scores else float('inf')
        
        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
        
        self.best_params = study.best_params
        print(f"   CatBoost 최적 RMSE: {study.best_value:.4f}")
        print(f"   최적 파라미터: {self.best_params}")
        return study.best_value

    def fit(self, df, target_col='heat_demand'):
        print(f"CatBoost 모델 훈련 중...")
        
        # ✅ 결측치 전처리 제거 - 원본 데이터 그대로 사용
        X = df[self.feature_cols].copy()
        y = df[target_col].copy()

        # ✅ 범주형 변수만 문자열로 변환 (결측치 채우기 없음)
        feature_config = model_features[f'catboost_{self.season_type}']
        categorical_features = [col for col in feature_config['categorical'] if col in df.columns]
        
        for col in categorical_features:
            if col in X.columns:
                X[col] = X[col].astype(str)

        # 최적화된 파라미터 사용
        self.model = CatBoostRegressor(
            iterations=self.best_params.get('iterations', 1000),
            learning_rate=self.best_params.get('learning_rate', 0.1),
            depth=self.best_params.get('depth', 6),
            l2_leaf_reg=self.best_params.get('l2_leaf_reg', 3),
            border_count=self.best_params.get('border_count', 128),
            bagging_temperature=self.best_params.get('bagging_temperature', 1),
            random_strength=self.best_params.get('random_strength', 1),
            leaf_estimation_iterations=self.best_params.get('leaf_estimation_iterations', 5),
            cat_features=self.categorical_features,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False
        )

        self.model.fit(X, y)
        print(f"   CatBoost 훈련 완료")

    def predict(self, df):
        # ✅ 결측치 전처리 제거 - 원본 데이터 그대로 사용
        X = df[self.feature_cols].copy()
        
        # ✅ 범주형 변수만 문자열로 변환 (결측치 채우기 없음)
        feature_config = model_features[f'catboost_{self.season_type}']
        categorical_features = [col for col in feature_config['categorical'] if col in df.columns]
        
        for col in categorical_features:
            if col in X.columns:
                X[col] = X[col].astype(str)
        
        return self.model.predict(X)

In [ ]:
class LSTMOptimizedModel:
    def __init__(self, season_type="heating"):
        self.model = None
        self.scaler = StandardScaler()
        self.device = device
        self.feature_cols = None
        self.best_params = {}
        self.season_type = season_type
        
    def optimize_hyperparameters(self, df, target_col='heat_demand', n_trials=30):
        print(f"LSTM 하이퍼파라미터 최적화 중... (trials: {n_trials})")
        
        # ARIMA 보간 적용
        df_interpolated = apply_arima_interpolation(df, target_col)
        
        # 피쳐 준비
        feature_config = model_features['lstm']
        self.feature_cols = [col for col in feature_config['numerical'] if col in df_interpolated.columns]
        
        # 브랜치 인코딩
        branch_encoder = LabelEncoder()
        df_interpolated['branch_encoded'] = branch_encoder.fit_transform(df_interpolated['branch_id'])
        self.feature_cols.append('branch_encoded')
        
        X = df_interpolated[self.feature_cols].values
        y = df_interpolated[target_col].values
        
        X = np.nan_to_num(X, nan=0)
        y = np.nan_to_num(y, nan=0)
        
        def objective(trial):
            # 하이퍼파라미터 샘플링
            hidden_size = trial.suggest_int('hidden_size', 64, 256)
            num_layers = trial.suggest_int('num_layers', 1, 3)
            dropout = trial.suggest_float('dropout', 0.1, 0.5)
            learning_rate = trial.suggest_float('learning_rate', 0.0001, 0.01, log=True)
            batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
            sequence_length = trial.suggest_int('sequence_length', 12, 48)
            
            # 3-Fold CV
            # 수정된 코드
            tscv = TimeSeriesSplit(n_splits=3)
            cv_scores = []
            
            # 시간순 정렬 보장
            df_sorted = df_interpolated.sort_values(['branch_id', 'tm']).reset_index(drop=True)
            X_sorted = df_sorted[self.feature_cols].values
            y_sorted = df_sorted[target_col].values
            X_sorted = np.nan_to_num(X_sorted, nan=0)
            y_sorted = np.nan_to_num(y_sorted, nan=0)

            for train_idx, val_idx in tscv.split(X_sorted):
                X_train, X_val = X_sorted[train_idx], X_sorted[val_idx]
                y_train, y_val = y_sorted[train_idx], y_sorted[val_idx]
                
                try:
                    # 스케일링
                    scaler = StandardScaler()
                    X_train_scaled = scaler.fit_transform(X_train)
                    X_val_scaled = scaler.transform(X_val)
                    
                    # 데이터셋 생성
                    train_dataset = TimeSeriesDataset(X_train_scaled, y_train, sequence_length)
                    val_dataset = TimeSeriesDataset(X_val_scaled, y_val, sequence_length)
                    
                    if len(train_dataset) == 0 or len(val_dataset) == 0:
                        continue
                        
                    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
                    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
                    
                    # 모델 생성
                    model = LSTMNet(
                        input_size=X_train_scaled.shape[1],
                        hidden_size=hidden_size,
                        num_layers=num_layers,
                        dropout=dropout
                    ).to(self.device)
                    
                    criterion = nn.MSELoss()
                    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
                    
                    # 훈련 (간단히)
                    model.train()
                    for epoch in range(10):  # 최적화를 위해 짧게
                        for batch_x, batch_y in train_loader:
                            batch_x = batch_x.to(self.device)
                            batch_y = batch_y.to(self.device)
                            
                            optimizer.zero_grad()
                            outputs = model(batch_x)
                            loss = criterion(outputs, batch_y)
                            loss.backward()
                            optimizer.step()
                    
                    # 검증
                    model.eval()
                    val_predictions = []
                    val_targets = []
                    
                    with torch.no_grad():
                        for batch_x, batch_y in val_loader:
                            batch_x = batch_x.to(self.device)
                            batch_y = batch_y.to(self.device)
                            
                            outputs = model(batch_x)
                            val_predictions.extend(outputs.cpu().numpy().flatten())
                            val_targets.extend(batch_y.cpu().numpy().flatten())
                    
                    if len(val_predictions) > 0:
                        rmse = np.sqrt(mean_squared_error(val_targets, val_predictions))
                        cv_scores.append(rmse)
                        
                except Exception as e:
                    continue
                    
            return np.mean(cv_scores) if cv_scores else 999.0
        
        # Optuna 스터디
        study = optuna.create_study(
            direction='minimize',
            sampler=TPESampler(seed=SEED)
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
        
        self.best_params = study.best_params
        print(f"   LSTM 최적 RMSE: {study.best_value:.4f}")
        print(f"   최적 파라미터: {self.best_params}")
        
        return study.best_value

    def fit(self, df, target_col='heat_demand'):
        print(f"LSTM 모델 훈련 중...")
        
        # ARIMA 보간 적용
        df_interpolated = apply_arima_interpolation(df, target_col)
        
        # 피쳐 준비 (optimize에서 이미 설정된 경우)
        if self.feature_cols is None:
            feature_config = model_features['lstm']
            self.feature_cols = [col for col in feature_config['numerical'] if col in df_interpolated.columns]
            
            # 브랜치 인코딩
            branch_encoder = LabelEncoder()
            df_interpolated['branch_encoded'] = branch_encoder.fit_transform(df_interpolated['branch_id'])
            self.feature_cols.append('branch_encoded')

        X = df_interpolated[self.feature_cols].values
        y = df_interpolated[target_col].values

        X = np.nan_to_num(X, nan=0)
        y = np.nan_to_num(y, nan=0)

        X_scaled = self.scaler.fit_transform(X)

        # 최적화된 파라미터 사용
        sequence_length = self.best_params.get('sequence_length', 24)
        dataset = TimeSeriesDataset(X_scaled, y, sequence_length)
        dataloader = DataLoader(
            dataset, 
            batch_size=self.best_params.get('batch_size', 64), 
            shuffle=True
        )

        if len(dataset) == 0:
            print("   데이터셋이 비어있습니다.")
            return

        self.model = LSTMNet(
            input_size=X_scaled.shape[1],
            hidden_size=self.best_params.get('hidden_size', 128),
            num_layers=self.best_params.get('num_layers', 2),
            dropout=self.best_params.get('dropout', 0.2)
        ).to(self.device)
        
        criterion = nn.MSELoss()
        optimizer = optim.Adam(
            self.model.parameters(), 
            lr=self.best_params.get('learning_rate', 0.001)
        )

        self.model.train()
        for epoch in tqdm(range(50), desc="LSTM 훈련"):
            for batch_x, batch_y in dataloader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)

                optimizer.zero_grad()
                outputs = self.model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
        print(f"   LSTM 훈련 완료")

    def predict(self, df):
        if self.model is None:
            return np.full(len(df), 0)

        # ARIMA 보간 적용
        df_interpolated = apply_arima_interpolation(df)
        
        # 브랜치 인코딩 (fit에서와 동일하게)
        if 'branch_encoded' not in df_interpolated.columns:
            branch_encoder = LabelEncoder()
            branch_encoder.fit(df_interpolated['branch_id'])
            df_interpolated['branch_encoded'] = branch_encoder.transform(df_interpolated['branch_id'])
        
        X = df_interpolated[self.feature_cols].values
        X = np.nan_to_num(X, nan=0)
        X_scaled = self.scaler.transform(X)

        self.model.eval()
        predictions = []
        sequence_length = self.best_params.get('sequence_length', 24)

        with torch.no_grad():
            for i in range(len(X_scaled)):
                if i < sequence_length:
                    predictions.append(0)
                else:
                    seq_data = X_scaled[i-sequence_length+1:i+1]
                    seq_tensor = torch.FloatTensor(seq_data).unsqueeze(0).to(self.device)
                    pred = self.model(seq_tensor).cpu().numpy()[0, 0]
                    predictions.append(pred)

        return np.array(predictions)

print("LSTM 최적화 모델 클래스 정의 완료")

## 7. 스태킹 앙상블 클래스 (Ridge 메타모델 최적화)

In [ ]:
class AdvancedStackingEnsemble:
    def __init__(self, season_type="heating", group_name=""):
        self.season_type = season_type
        self.group_name = group_name
        self.models = {
            'prophet': ProphetOptimizedModel(season_type),
            'catboost': CatBoostOptimizedModel(season_type),
            'lstm': LSTMOptimizedModel(season_type)
        }
        self.meta_model = None
        self.best_meta_params = {}
        self.individual_scores = {}

    def optimize_meta_model(self, level1_features, targets, n_trials=20):
        print(f"Ridge 메타모델 최적화 중... (trials: {n_trials})")
        
        def objective(trial):
            alpha = trial.suggest_float('alpha', 0.01, 100, log=True)
            
            # 3-Fold CV
            tscv = TimeSeriesSplit(n_splits=3)
            cv_scores = []
            
            for train_idx, val_idx in tscv.split(level1_features):
                X_train, X_val = level1_features[train_idx], level1_features[val_idx]
                y_train, y_val = targets[train_idx], targets[val_idx]
                
                try:
                    model = Ridge(alpha=alpha, random_state=SEED)
                    model.fit(X_train, y_train)
                    pred = model.predict(X_val)
                    rmse = np.sqrt(mean_squared_error(y_val, pred))
                    cv_scores.append(rmse)
                except Exception as e:
                    continue
                    
            return np.mean(cv_scores) if cv_scores else 999.0
        
        # Optuna 스터디
        study = optuna.create_study(
            direction='minimize',
            sampler=TPESampler(seed=SEED)
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
        
        self.best_meta_params = study.best_params
        print(f"   Ridge 최적 RMSE: {study.best_value:.4f}")
        print(f"   최적 파라미터: {self.best_meta_params}")
        
        return study.best_value

    def fit(self, train_df, target_col='heat_demand', optimize_trials=None):
        print(f"\n{self.group_name} 스태킹 앙상블 훈련 시작!")
        print("=" * 60)
        
        if len(train_df) < 50:
            print(f"데이터가 부족합니다 ({len(train_df)}개). 스킵합니다.")
            return
            
        # 기본 trials 설정
        if optimize_trials is None:
            optimize_trials = {'prophet': 30, 'catboost': 50, 'lstm': 30, 'meta': 20}

        # 브랜치별 시간 기반 분할로 검증 데이터 생성
        train_data_list = []
        val_data_list = []

        for branch in train_df['branch_id'].unique():
            branch_data = train_df[train_df['branch_id'] == branch].copy().sort_values('tm')
            val_size = max(1, int(len(branch_data) * 0.2))

            train_data_list.append(branch_data.iloc[:-val_size])
            val_data_list.append(branch_data.iloc[-val_size:])

        train_fit_df = pd.concat(train_data_list, ignore_index=True)
        val_df = pd.concat(val_data_list, ignore_index=True)

        print(f"훈련: {len(train_fit_df):,}개, 검증: {len(val_df):,}개")

        # 1단계: 개별 모델 하이퍼파라미터 최적화 및 훈련
        level1_predictions = {}

        for name, model in self.models.items():
            print(f"\n{name.upper()} 최적화 및 훈련...")
            try:
                start_time = datetime.now()
                
                # 하이퍼파라미터 최적화
                best_score = model.optimize_hyperparameters(
                    train_fit_df, target_col, n_trials=optimize_trials[name]
                )
                
                # 최적화된 파라미터로 전체 훈련 데이터에 재훈련
                model.fit(train_fit_df, target_col)
                
                # 검증 데이터 예측
                val_pred = model.predict(val_df)
                level1_predictions[name] = val_pred

                # 개별 모델 성능 계산
                rmse = np.sqrt(mean_squared_error(val_df[target_col], val_pred))
                mae = mean_absolute_error(val_df[target_col], val_pred)
                train_time = (datetime.now() - start_time).total_seconds()

                self.individual_scores[name] = {
                    'rmse': rmse, 
                    'mae': mae, 
                    'optuna_score': best_score,
                    'train_time': train_time
                }

                print(f"   {name} 성능: RMSE={rmse:.4f}, MAE={mae:.4f}")
                print(f"   Optuna 최적 점수: {best_score:.4f}")
                print(f"   총 시간: {train_time:.1f}초")

            except Exception as e:
                print(f"   {name} 훈련 실패: {str(e)[:100]}...")
                level1_predictions[name] = np.full(len(val_df), val_df[target_col].mean())
                self.individual_scores[name] = {'rmse': 999, 'mae': 999, 'optuna_score': 999, 'train_time': 0}

        # 2단계: 메타 모델 최적화 및 훈련
        print(f"\nRidge 메타 모델 최적화 및 훈련...")
        meta_features = np.column_stack(list(level1_predictions.values()))
        
        # 메타모델 하이퍼파라미터 최적화
        meta_score = self.optimize_meta_model(meta_features, val_df[target_col].values, optimize_trials['meta'])
        
        # 최적화된 파라미터로 메타모델 훈련
        self.meta_model = Ridge(
            alpha=self.best_meta_params.get('alpha', 1.0),
            random_state=SEED
        )
        self.meta_model.fit(meta_features, val_df[target_col])

        # 스태킹 성능
        stacking_pred = self.meta_model.predict(meta_features)
        stacking_rmse = np.sqrt(mean_squared_error(val_df[target_col], stacking_pred))
        stacking_mae = mean_absolute_error(val_df[target_col], stacking_pred)

        self.individual_scores['stacking'] = {
            'rmse': stacking_rmse, 
            'mae': stacking_mae,
            'optuna_score': meta_score
        }
        
        print(f"   스태킹 성능: RMSE={stacking_rmse:.4f}, MAE={stacking_mae:.4f}")
        print(f"✅ {self.group_name} 스태킹 앙상블 훈련 완료!")

    def predict(self, test_df):
        """스태킹 앙상블 예측"""
        if self.meta_model is None:
            print(f"⚠️ {self.group_name} 모델이 훈련되지 않았습니다.")
            return np.full(len(test_df), 0), {}

        level1_predictions = {}

        # 1단계: 개별 모델 예측
        for name, model in self.models.items():
            try:
                level1_predictions[name] = model.predict(test_df)
            except Exception as e:
                print(f"❌ {name} 예측 실패: {e}")
                level1_predictions[name] = np.full(len(test_df), 0)

        # 2단계: 메타 모델 예측
        meta_features = np.column_stack(list(level1_predictions.values()))
        final_pred = self.meta_model.predict(meta_features)

        return final_pred, level1_predictions

print("✅ 고도화된 스태킹 앙상블 클래스 정의 완료")

# 결과 저장용 딕셔너리
ensemble_models = {}
group_results = {}

print("\n🎯 6개 그룹별 개별 훈련 준비 완료!")

## 8. 그룹별 개별 훈련

In [ ]:
# 모든 그룹 훈련 함수
def train_all_groups():
    """2개 그룹 훈련 (난방/비난방)"""
    group_configs = {
        "heating": {
            "season": "heating", 
            "trials": {"prophet": 50, "catboost": 80, "lstm": 50, "meta": 30}
        },
        "non_heating": {
            "season": "non_heating", 
            "trials": {"prophet": 40, "catboost": 60, "lstm": 40, "meta": 25}
        }
    }
    
    print("🚀 시즌별 2개 그룹 훈련 시작!")
    total_start_time = datetime.now()
    
    for group_name, config in group_configs.items():
        print(f"\n{'='*60}")
        print(f"🔥 {group_name.upper()} 그룹 훈련")
        print(f"📊 데이터 크기: {len(train_groups[group_name]):,}개")
        print(f"🏢 지사 수: {train_groups[group_name]['branch_id'].nunique()}개")
        
        if len(train_groups[group_name]) > 0:
            # 앙상블 모델 생성
            ensemble_models[group_name] = AdvancedStackingEnsemble(
                season_type=config["season"], 
                group_name=group_name
            )
            
            # 훈련 실행
            start_time = datetime.now()
            ensemble_models[group_name].fit(
                train_groups[group_name], 
                target_col='heat_demand',
                optimize_trials=config["trials"]
            )
            total_time = (datetime.now() - start_time).total_seconds()
            
            # 결과 저장
            group_results[group_name] = {
                'scores': ensemble_models[group_name].individual_scores.copy(),
                'total_time': total_time,
                'data_size': len(train_groups[group_name])
            }
            
            print(f"\n📈 {group_name} 최종 결과:")
            for model, scores in ensemble_models[group_name].individual_scores.items():
                rmse = scores.get('rmse', 999)
                mae = scores.get('mae', 999)
                print(f"   {model:12s}: RMSE={rmse:.4f}, MAE={mae:.4f}")
            print(f"   ⏱️ 총 훈련 시간: {total_time:.1f}초")
            
        else:
            print(f"⚠️ {group_name} 데이터가 없습니다.")
            group_results[group_name] = None
    
    total_training_time = (datetime.now() - total_start_time).total_seconds()
    print(f"\n🎉 전체 훈련 완료! 총 시간: {total_training_time/60:.1f}분")

# 모든 그룹 훈련 실행
train_all_groups()

## 9. 전체 그룹 결과 요약

In [ ]:
# 전체 그룹 훈련 결과 요약
print("\n🏆 전체 그룹 훈련 결과 요약")
print("=" * 80)

total_time = 0
total_data_size = 0
successful_groups = 0

print(f"{'그룹명':20s} {'데이터':>8s} {'Prophet':>8s} {'CatBoost':>9s} {'LSTM':>8s} {'Stacking':>9s} {'시간(분)':>8s}")
print("-" * 80)

for group_name, result in group_results.items():
    if result is not None:
        scores = result['scores']
        data_size = result['data_size']
        group_time = result['total_time']
        
        total_time += group_time
        total_data_size += data_size
        successful_groups += 1
        
        prophet_rmse = scores.get('prophet', {}).get('rmse', 999)
        catboost_rmse = scores.get('catboost', {}).get('rmse', 999)
        lstm_rmse = scores.get('lstm', {}).get('rmse', 999)
        stacking_rmse = scores.get('stacking', {}).get('rmse', 999)
        
        print(f"{group_name:20s} {data_size:8,d} {prophet_rmse:8.4f} {catboost_rmse:9.4f} {lstm_rmse:8.4f} {stacking_rmse:9.4f} {group_time/60:8.1f}")
    else:
        print(f"{group_name:20s} {'N/A':>8s} {'N/A':>8s} {'N/A':>9s} {'N/A':>8s} {'N/A':>9s} {'N/A':>8s}")

print("-" * 80)
print(f"{'TOTAL':20s} {total_data_size:8,d} {'':>8s} {'':>9s} {'':>8s} {'':>9s} {total_time/60:8.1f}")
print(f"\n✅ 성공한 그룹: {successful_groups}/6")
print(f"⏱️ 총 훈련 시간: {total_time/60:.1f}분 ({total_time/3600:.1f}시간)")

# 그룹별 최고 성능 모델 찾기
print(f"\n🥇 그룹별 최고 성능 모델:")
for group_name, result in group_results.items():
    if result is not None:
        scores = result['scores']
        best_model = min(
            [(name, score['rmse']) for name, score in scores.items() 
             if isinstance(score, dict) and 'rmse' in score],
            key=lambda x: x[1],
            default=("None", 999)
        )
        print(f"   {group_name:20s}: {best_model[0].upper():10s} (RMSE: {best_model[1]:.4f})")

# 모델별 평균 성능
print(f"\n📊 모델별 평균 성능:")
model_avg_scores = {'prophet': [], 'catboost': [], 'lstm': [], 'stacking': []}

for result in group_results.values():
    if result is not None:
        for model_name in model_avg_scores.keys():
            if (model_name in result['scores'] and 
                isinstance(result['scores'][model_name], dict) and 
                'rmse' in result['scores'][model_name]):
                model_avg_scores[model_name].append(result['scores'][model_name]['rmse'])

for model_name, scores in model_avg_scores.items():
    if scores:
        avg_score = np.mean(scores)
        std_score = np.std(scores)
        print(f"   {model_name.upper():12s}: {avg_score:.4f} (±{std_score:.4f})")

## 🔟 테스트 데이터 예측

In [ ]:
# 테스트 데이터 예측
print("🎯 테스트 데이터 예측 시작...")

# 예측 결과 저장용
test_predictions = {}
individual_predictions = {}

# 최종 예측 결과 통합에서도 2개 그룹만 처리
for group_name in ['heating', 'non_heating']:
    if group_name in ensemble_models and len(test_groups[group_name]) > 0:
        print(f"\n📊 {group_name} 예측 중...")
        
        try:
            pred, individual_pred = ensemble_models[group_name].predict(test_groups[group_name])
            test_predictions[group_name] = pred
            individual_predictions[group_name] = individual_pred
            
            print(f"   ✅ {group_name}: {len(pred):,}개 예측 완료")
            print(f"   📈 예측값 범위: {pred.min():.2f} ~ {pred.max():.2f}")
            print(f"   📊 예측값 평균: {pred.mean():.2f}")
            
        except Exception as e:
            print(f"   ❌ {group_name} 예측 실패: {str(e)[:100]}...")
            test_predictions[group_name] = np.zeros(len(test_groups[group_name]))
    else:
        if len(test_groups[group_name]) > 0:
            print(f"⚠️ {group_name}: 훈련된 모델 없음, 0으로 채움")
            test_predictions[group_name] = np.zeros(len(test_groups[group_name]))

print("\n✅ 모든 그룹 예측 완료!")

## 1️⃣1️⃣ 최종 결과 통합 및 저장

In [ ]:
# 최종 예측 결과 통합
print("💾 최종 예측 결과 통합 및 저장...")

# 기본 결과 데이터프레임 생성
result_df = test_df[['tm', 'branch_id', 'heating_season', 'size_group']].copy()

# 그룹별 예측 결과 통합
final_stacking_pred = np.zeros(len(test_df))
final_prophet_pred = np.zeros(len(test_df))
final_catboost_pred = np.zeros(len(test_df))
final_lstm_pred = np.zeros(len(test_df))

print("📊 그룹별 예측 결과 통합 중...")

# 각 그룹별로 해당하는 인덱스에 예측값 할당
for group_name, group_data in test_groups.items():
    if len(group_data) > 0 and group_name in test_predictions:
        group_indices = group_data.index
        group_pred = test_predictions[group_name]
        
        print(f"   {group_name}: {len(group_indices)}개 인덱스, {len(group_pred)}개 예측값")
        
        # 인덱스 길이 맞추기
        min_length = min(len(group_indices), len(group_pred))
        if min_length > 0:
            final_stacking_pred[group_indices[:min_length]] = group_pred[:min_length]
            
            # 개별 모델 예측값도 저장
            if group_name in individual_predictions:
                individual_pred = individual_predictions[group_name]
                
                if 'prophet' in individual_pred and len(individual_pred['prophet']) >= min_length:
                    final_prophet_pred[group_indices[:min_length]] = individual_pred['prophet'][:min_length]
                if 'catboost' in individual_pred and len(individual_pred['catboost']) >= min_length:
                    final_catboost_pred[group_indices[:min_length]] = individual_pred['catboost'][:min_length]
                if 'lstm' in individual_pred and len(individual_pred['lstm']) >= min_length:
                    final_lstm_pred[group_indices[:min_length]] = individual_pred['lstm'][:min_length]

# 음수값 제거
final_stacking_pred = np.maximum(final_stacking_pred, 0)
final_prophet_pred = np.maximum(final_prophet_pred, 0)
final_catboost_pred = np.maximum(final_catboost_pred, 0)
final_lstm_pred = np.maximum(final_lstm_pred, 0)

# 결과 데이터프레임에 추가
result_df['stacking_prediction'] = final_stacking_pred.round(1)
result_df['prophet_prediction'] = final_prophet_pred.round(1)
result_df['catboost_prediction'] = final_catboost_pred.round(1)
result_df['lstm_prediction'] = final_lstm_pred.round(1)

# 통계 출력
print(f"\n📈 최종 예측값 통계:")
prediction_cols = ['stacking_prediction', 'prophet_prediction', 'catboost_prediction', 'lstm_prediction']
for col in prediction_cols:
    mean_val = result_df[col].mean()
    std_val = result_df[col].std()
    max_val = result_df[col].max()
    min_val = result_df[col].min()
    print(f"   {col:20s}: 평균={mean_val:7.1f}, 표준편차={std_val:6.1f}, 범위=[{min_val:.1f}, {max_val:.1f}]")

# CSV 파일 저장
result_filename = 'advanced_stacking_ensemble_predictions.csv'
result_df.to_csv(result_filename, index=False)
print(f"\n📁 상세 예측 결과 저장: {result_filename}")

# 제출용 파일 생성 (스태킹 앙상블 결과만)
submission_df = test_df[['tm', 'branch_id']].copy()
submission_df['heat_demand'] = result_df['stacking_prediction']

submission_filename = 'submission_advanced_stacking.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"📁 제출용 파일 저장: {submission_filename}")

# 그룹별 예측 통계
print(f"\n📊 그룹별 예측 통계 (스태킹 모델):")
try:
    group_stats = result_df.groupby(['heating_season', 'size_group'])['stacking_prediction'].agg([
        'count', 'mean', 'std', 'min', 'max'
    ]).round(2)
    print(group_stats)
except Exception as e:
    print(f"   그룹별 통계 계산 실패: {e}")

# Google Drive 저장 (Colab 환경)
if IN_COLAB:
    try:
        save_drive = input("\nGoogle Drive에 저장하시겠습니까? (y/n): ").lower().strip()
        if save_drive == 'y':
            os.system(f"cp {result_filename} /content/drive/MyDrive/")
            os.system(f"cp {submission_filename} /content/drive/MyDrive/")
            print("✅ Google Drive 저장 완료!")
    except Exception as e:
        print(f"⚠️ Google Drive 저장 중 오류: {e}")

print("\n🎊 모든 작업 완료!")

## 1️⃣2️⃣ 최종 분석 요약

In [ ]:
print("\n📋 🔥 최종 분석 요약 🔥")
print("=" * 80)

print(f"✅ 모델 구성:")
print(f"   🎯 고도화된 스태킹 앙상블 (Prophet + CatBoost + LSTM + Ridge)")
print(f"   📊 그룹별 전용 모델: 3개 규모 × 2개 시즌 = 6개 그룹")
print(f"   🔬 총 모델 수: 18개 (6그룹 × 3모델) + 6개 메타모델 = 24개")
print(f"   🎲 완전한 재현성: 모든 시드 고정 (SEED={SEED})")

print(f"\n🔧 기술적 혁신:")
print(f"   🌡️ 시즌별 도메인 특화 이상치 플래그")
print(f"   📈 모델별 차별화된 피쳐 전략")
print(f"   🎯 Optuna TPE + 3-Fold CV로 전 모델 최적화")
print(f"   🔄 ARIMA 시계열 보간 (LSTM용)")

print(f"\n📊 모델별 특화 전략:")
print(f"   🔮 Prophet: 시계열 중심, 결측치 내장 처리")
print(f"   🐱 CatBoost: 범주형 최적화, 결측치+이상치 플래그")
print(f"   🧠 LSTM: 수치형 정규화, ARIMA 보간")
print(f"   🎯 Ridge: 최적화된 메타모델")

# 최종 성능 요약
if group_results:
    successful_groups = sum(1 for result in group_results.values() if result is not None)
    total_training_time = sum(
        result['total_time'] for result in group_results.values() 
        if result is not None
    )
    
    print(f"\n🏆 훈련 결과:")
    print(f"   ✅ 성공한 그룹: {successful_groups}/6")
    print(f"   ⏱️ 총 훈련 시간: {total_training_time/60:.1f}분")
    
    # 스태킹 vs 개별 모델 성능 비교
    stacking_scores = []
    individual_scores = {'prophet': [], 'catboost': [], 'lstm': []}
    
    for result in group_results.values():
        if result is not None and 'scores' in result:
            scores = result['scores']
            if ('stacking' in scores and 
                isinstance(scores['stacking'], dict) and 
                'rmse' in scores['stacking']):
                stacking_scores.append(scores['stacking']['rmse'])
            
            for model in individual_scores.keys():
                if (model in scores and 
                    isinstance(scores[model], dict) and 
                    'rmse' in scores[model]):
                    individual_scores[model].append(scores[model]['rmse'])
    
    if stacking_scores:
        print(f"\n📈 평균 성능 (RMSE):")
        for model, scores in individual_scores.items():
            if scores:
                avg_score = np.mean(scores)
                print(f"   {model.upper():12s}: {avg_score:.4f}")
        
        stacking_avg = np.mean(stacking_scores)
        print(f"   {'STACKING':12s}: {stacking_avg:.4f} ⭐")
        
        # 개선율 계산
        individual_avgs = [np.mean(scores) for scores in individual_scores.values() if scores]
        if individual_avgs:
            best_individual = min(individual_avgs)
            improvement = (best_individual - stacking_avg) / best_individual * 100
            print(f"\n🎯 스태킹 개선율: {improvement:.1f}%")

print(f"\n📁 출력 파일:")
print(f"   📊 상세 결과: advanced_stacking_ensemble_predictions.csv")
print(f"   🏆 제출용: submission_advanced_stacking.csv")

print(f"\n🎉 지역난방 열수요 예측 완료!")
print(f"🚀 세계 최고 수준의 고도화된 스태킹 앙상블 적용 성공!")

# GPU 메모리 정리
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"🧹 GPU 메모리 정리 완료")

print(f"\n💡 재현성 보장: 동일한 시드({SEED})로 언제든 동일한 결과 재현 가능")
print(f"🎯 Data Leakage 방지를 위해 추가 수정이 필요할 수 있습니다.")